# 03 Weather data quality

Evidence for the decisions taken in notebook 02: that all four daily variables
share one grid, that the monthly files tile the window without gaps, and what
observing window each variable declares.

Downloads one month of each variable, about 290 MB, and deletes them at the end.
Each check prints a verdict.

In [ ]:
import numpy as np
import pandas as pd

from fdh import weather as wx
from fdh import weather_checks as wc
from fdh.config import load_config, require

config = load_config()
BASE = require(config.weather.base_url, "weather.base_url")
VERSION = require(config.weather.version_directory, "weather.version_directory")
START = config.weather.window_start
END = config.weather.release_covers_to
print("token set:", wx.ensure_token())

findings = []          # one row per check, written to the quality report at the end
def record(check, verdict, detail):
    findings.append({"check": check, "verdict": verdict, "detail": detail})
    print(f"{verdict:6s}  {check}: {detail}")

## 1. Which variables publish daily

The panel needs daily values.

In [ ]:
daily = wx.daily_variables(BASE)
VARIABLES = sorted(name for name, has_day in daily.items() if has_day)
configured = require(config.weather.variables, "weather.variables")
not_daily = sorted(set(configured) - set(VARIABLES))

record("daily variables",
       "FAIL" if not_daily else "PASS",
       f"{len(VARIABLES)} of {len(daily)} publish daily: {VARIABLES}; "
       + (f"configured but not daily: {not_daily}" if not_daily
          else f"every configured variable {configured} publishes daily"))
pd.Series(daily, name="has_daily").to_frame()

## 2. One version directory per variable

In [ ]:
versions = {name: wx.list_directories(f"{BASE}/{name}/day/") for name in VARIABLES}
as_configured = all(found == [f"{VERSION}/"] for found in versions.values())

record("version directories",
       "PASS" if as_configured else "CHECK",
       f"weather.version_directory {VERSION}; found {versions}")

## 3. The monthly files tile the window

Gaps mean missing weather days. Overlaps mean a day counted twice.

In [ ]:
files = {name: wx.daily_files(BASE, name, VERSION, START, END) for name in VARIABLES}
coverage = pd.DataFrame(wc.contiguity_rows(files))
expected = wc.expected_months(START, END)
clean = (coverage.gaps.sum() == 0 and coverage.overlaps.sum() == 0
         and (coverage.distinct_months == coverage.files).all()
         and (coverage.distinct_months == expected).all())

record("file contiguity",
       "PASS" if clean else "CHECK",
       f"{expected} months expected per variable; gaps {coverage.gaps.sum()}, "
       f"overlaps {coverage.overlaps.sum()}, duplicate months "
       f"{(coverage.files - coverage.distinct_months).sum()}, "
       f"months per variable {sorted(set(coverage.distinct_months))}")
coverage

## 4. Every variable offers the same months

An unbalanced panel would silently shorten whichever covariate is missing.

In [ ]:
balance = pd.DataFrame(wc.coverage_rows(files))
short = balance.loc[balance.missing.map(len) > 0]

record("month coverage",
       "PASS" if short.empty else "CHECK",
       f"{balance.months.max()} months offered across {len(VARIABLES)} variables"
       + ("; every variable offers all of them" if short.empty
          else f"; missing {dict(zip(short.variable, short.missing, strict=True))}"))
balance

## 5. Pull one month of each variable

The first month of the weather window, `weather.window_start` in config. The
files are deleted in the last cell.

In [ ]:
probe_dir = config.paths.resolved("raw_dir") / "weather" / "probe"
probes = {}
for name, items in files.items():
    probes[name] = wx.download(items[0], probe_dir)
    print(f"{name}: {probes[name].name} {probes[name].stat().st_size / 1e6:.0f} MB")

described = {name: wc.describe_file(path) for name, path in probes.items()}

## 6. The declared observing window

The window decides how a weather day aligns with an incident day. Each file
states it twice: in `time_bnds`, and in the text of `cell_methods` and `comment`.
A file whose text says nothing about a 09:00 day is reported as not stated, not
as a disagreement.

In [ ]:
windows = pd.DataFrame(wc.window_rows(list(described.values())))
disagree = windows.loc[windows.agreement == "disagree", "variable"].tolist()
unstated = windows.loc[windows.agreement == "not stated", "variable"].tolist()
one_window = windows.bounds_start_hours.astype(str).nunique() == 1

record("observing window",
       "PASS" if one_window and not disagree else "CHECK",
       ("one window across variables" if one_window else "variables declare different windows")
       + (f"; bounds and text disagree for {disagree}" if disagree else "")
       + (f"; text states no window for {unstated}" if unstated else ""))
windows

## 7. One grid for all four variables

The cell-to-service mask is built once and applied to every file.

In [ ]:
grids = pd.DataFrame(wc.grid_rows(list(described.values())))
reference = described[VARIABLES[0]]
aligned = grids.x_matches.all() and grids.y_matches.all()

record("grid alignment",
       "PASS" if aligned else "FAIL",
       f"shape {reference['shape'][1:]}, {reference['valid_cells']:,} valid cells "
       f"({reference['valid_share']:.1%}); masks identical: {grids.mask_matches.all()}")
grids

## 8. Metadata contract and value ranges

Provenance strings go into the methodology. The ranges are a sanity check on
units.

In [ ]:
values = pd.DataFrame(wc.range_rows(list(described.values())))
provenance_consistent = values.source.nunique() == 1 and values.version.nunique() == 1
within = values.within_expected.eq(True).all()

record("metadata contract",
       "PASS" if provenance_consistent and within else "CHECK",
       f"source {values.source.iloc[0]}, version {values.version.iloc[0]}; "
       f"values within expected range: {within}")
values

## 9. Cross-variable consistency

`tasmax` labelled D covers 09:00 on D to 09:00 on D+1 and `tasmin` labelled D
covers 09:00 on D-1 to 09:00 on D (Hollis et al., 2019, Table 1), as their
`cell_methods` state. The check stops if either file states another period.

Same window: `tasmin` labelled D+1 against `tasmax` labelled D. Both cover 09:00
on D to 09:00 on D+1, so the minimum cannot exceed the maximum. Any violation is
an interpolation artefact, the two fields being interpolated onto the grid
separately. The share is reported with no verdict and no tolerance: the panel
takes a territory mean over at least several hundred cells, and the share is a
stated limitation of the gridded product rather than a condition of the build.

Different windows, reported alongside: `tasmin` and `tasmax` on one label D.
The minimum covers 09:00 on D-1 to 09:00 on D and the maximum the following 24
hours, so a minimum above the maximum is expected behaviour by definition.

In [ ]:
from netCDF4 import Dataset

stack = {}
for name, path in probes.items():
    with Dataset(path) as ds:
        var = next(v for v in ds.variables.values() if v.ndim == 3)
        stack[name] = np.ma.masked_invalid(var[:])

ordering = wc.temperature_ordering(
    stack["tasmin"],
    stack["tasmax"],
    described["tasmin"]["cell_methods"],
    described["tasmax"]["cell_methods"],
)
record("temperature ordering",
       "NOTE",
       f"same window (tasmin labelled D+1 and tasmax labelled D): minimum above "
       f"maximum in {ordering['failing']:,} of {ordering['cell_days']:,} cell-days "
       f"({ordering['failing_share']:.4%}); median excess "
       f"{ordering['median_excess']:.2f} C, max {ordering['max_excess']:.2f} C. "
       f"Different windows (both on one label D): {ordering['same_day_failing']:,} "
       f"of {ordering['same_day_cell_days']:,} cell-days, expected behaviour")
ordering

## 10. tas is a definition, not an independent field

cell_methods on tas is "time: mid_range", so the published mean is the midpoint
of tasmax and tasmin by construction. Comparing the two confirms that definition
holds in the data. Exact agreement is the expected result, and a difference would
mean the files do not match their metadata.

In [ ]:
valid = ~stack["tasmax"].mask & ~stack["tasmin"].mask & ~stack["tas"].mask
derived = (stack["tasmax"] + stack["tasmin"]) / 2
difference = (derived - stack["tas"])[valid]
exact = bool((np.abs(difference) < 1e-9).all())
record("tas is mid_range",
       "PASS" if exact else "FAIL",
       f"published tas equals (tasmax + tasmin) / 2 exactly: {exact}; "
       f"max absolute difference {float(np.abs(difference).max()):.2e} over "
       f"{valid.sum():,} cell-days. tas carries no information beyond the pair "
       f"and is dropped from the download.")

## Verdicts

In [ ]:
report = pd.DataFrame(findings)
report

In [ ]:
out = config.paths.resolved("reports_dir")
out.mkdir(parents=True, exist_ok=True)
report.to_csv(out / "weather_quality_checks.csv", index=False)
print(f"wrote {out / 'weather_quality_checks.csv'}")

## Clean up

The probe files are 290 MB and serve no further purpose.

In [ ]:
for path in probes.values():
    path.unlink(missing_ok=True)
print("probe files removed")